# Milestone 1: Week 2 - Lasso, Ridge, Elastic Net Regression

In [47]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import statsmodels.api as sm

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

seed = 42

#### Lasso

In [5]:
socioeconomic_indicators = pd.read_csv(r'..\Processed Data\USDA Socioeconomic Indicators.csv')
socioeconomic_indicators.sample(5)

,FIPS_Code,State,Area_Name,Year,Civilian_labor_force,Employed,Med_HH_Income_Percent_of_State_Total,Median_Household_Income,Metro,Rural_Urban_Continuum_Code,Unemployed,Unemployment_rate,Urban_Influence_Code
74991,55057,WI,"Juneau County, WI",2003,13117.0,12156.0,87.1,61904.0,0.0,8.0,961.0,7.326370,9.0
13552,16001,ID,"Ada County, ID",2014,210952.0,203035.0,120.8,87748.0,1.0,2.0,7917.0,3.752986,2.0
16446,17153,IL,"Pulaski County, IL",2004,2913.0,2659.0,58.5,44873.0,0.0,8.0,254.0,8.719533,7.0
55769,42083,PA,"McKean County, PA",2005,21791.0,20597.0,80.9,58112.0,0.0,7.0,1194.0,5.479326,8.0
1491,1121,AL,"Talladega County, AL",2003,38355.0,35403.0,92.4,55186.0,0.0,4.0,2952.0,7.696519,3.0


Using Lasso to get coefficients for predicting `Median_Household_Income`

In [39]:
# Getting numeric columns for Lasso regression

X = socioeconomic_indicators.select_dtypes(include='number').drop(columns=['FIPS_Code', 'Median_Household_Income']) # while FIPS_Code is technically numeric, I am dropping it since it is more categorical for location
y = socioeconomic_indicators['Median_Household_Income']

feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

# Testing different alpha scales
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1_000, 10_000]

lasso_cv = make_pipeline(StandardScaler(), LassoCV(alphas=alphas, cv=5, max_iter=100_000, random_state=seed))
lasso_cv.fit(X_train, y_train)

# Lasso model from pipeline
lasso_model = lasso_cv.named_steps['lassocv']

best_alpha = lasso_model.alpha_
print(f"Optimal Alpha: {best_alpha}\n")


print(f"Lasso coefficients with lambda = {best_alpha}:")
for name, coef in zip(feature_names, lasso_model.coef_):
    if abs(coef) > 1e-8:
        print(f"*** {name}: {coef: .6f}  <-- nonzero")
    else:
        print(f"    {name}: {coef: .6f}")

print("\nNumber of nonzero coefficients:", np.sum(np.abs(lasso_model.coef_) > 1e-8))


# Model evaluation
y_pred = lasso_cv.predict(X_test)
MAE = mean_absolute_error(y_test, y_pred)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R2 = r2_score(y_test, y_pred)


print("\nModel Evaluation")
print(f"*** Mean Absolute Error: ${MAE:,.2f}")
print(f"*** Root Mean Squared Error: ${RMSE:,.2f}")
print(f"*** R-squared Score: {R2:.4f}")

Optimal Alpha: 10.0

Lasso coefficients with lambda = 10.0:
*** Year: -161.068834  <-- nonzero
*** Civilian_labor_force:  61.692977  <-- nonzero
    Employed:  0.000000
*** Med_HH_Income_Percent_of_State_Total:  12836.351257  <-- nonzero
*** Metro:  34.284076  <-- nonzero
*** Rural_Urban_Continuum_Code: -1809.078529  <-- nonzero
*** Unemployed:  152.959898  <-- nonzero
*** Unemployment_rate: -1186.393508  <-- nonzero
*** Urban_Influence_Code:  111.347462  <-- nonzero

Number of nonzero coefficients: 8

Model Evaluation
*** Mean Absolute Error: $6,432.32
*** Root Mean Squared Error: $8,849.34
*** R-squared Score: 0.7117


Results:
  - Since `Employed` was dropped, it had high collinearity with other features. This is likely due to it being similar to `Civilian_labor_force`, which is the total `Employed` + total `Unemployed`.
  - `Med_HH_Income_Percent_of_State_Total` is the strongest positive driving force in this Lasso regression. This is likely because of data leakage since the median household income is used to calculate the percent of state total. I will rerun this Lasso regression with cross validation again with this feature dropped to get a clearer picture of the coefficients.

In [40]:
# Redoing LassoCV to prevent data leakage from Med_HH_Income_Percent_of_State_Total

X = socioeconomic_indicators.select_dtypes(include='number').drop(columns=['FIPS_Code', 'Median_Household_Income', 'Med_HH_Income_Percent_of_State_Total']) # while FIPS_Code is technically numeric, I am dropping it since it is more categorical for location
y = socioeconomic_indicators['Median_Household_Income']

feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

# Testing different alpha scales
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1_000, 10_000]

lasso_cv = make_pipeline(StandardScaler(), LassoCV(alphas=alphas, cv=5, max_iter=100_000, random_state=seed))
lasso_cv.fit(X_train, y_train)

# Lasso model from pipeline
lasso_model = lasso_cv.named_steps['lassocv']

best_alpha = lasso_model.alpha_
print(f"Optimal Alpha: {best_alpha}\n")


print(f"Lasso coefficients with lambda = {best_alpha}:")
for name, coef in zip(feature_names, lasso_model.coef_):
    if abs(coef) > 1e-8:
        print(f"*** {name}: {coef: .6f}  <-- nonzero")
    else:
        print(f"    {name}: {coef: .6f}")

print("\nNumber of nonzero coefficients:", np.sum(np.abs(lasso_model.coef_) > 1e-8))


# Model evaluation
y_pred = lasso_cv.predict(X_test)
MAE = mean_absolute_error(y_test, y_pred)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R2 = r2_score(y_test, y_pred)


print("\nModel Evaluation")
print(f"*** Mean Absolute Error: ${MAE:,.2f}")
print(f"*** Root Mean Squared Error: ${RMSE:,.2f}")
print(f"*** R-squared Score: {R2:.4f}")

Optimal Alpha: 10.0

Lasso coefficients with lambda = 10.0:
*** Year: -771.357105  <-- nonzero
    Civilian_labor_force:  0.000000
    Employed:  0.000000
*** Metro:  1091.561966  <-- nonzero
*** Rural_Urban_Continuum_Code: -5536.854461  <-- nonzero
*** Unemployed:  384.328665  <-- nonzero
*** Unemployment_rate: -5886.147130  <-- nonzero
*** Urban_Influence_Code: -537.709256  <-- nonzero

Number of nonzero coefficients: 6

Model Evaluation
*** Mean Absolute Error: $10,342.01
*** Root Mean Squared Error: $13,868.24
*** R-squared Score: 0.2919


Results:
  - `Civilian_labor_force` and `Employed` were dropped. This makes sense because we already capture that information through keeping the `Unemployed` and `Unemployment_rate`. Having both the employed and unemployed features would be redundant because
  - The strongest negative impacted features in predicting the `Median_Household_Income` were `Unemployment_rate`, `Rural_Urban_Continuum_Code`, and `Year`.
    - The `Unemployment_rate` being the most negative coefficient makes sense for the domain of this dataset because when there are higher rates of unemployment, the `Median_Household_Income` would decrease as the workforce shrinks.
    - `Rural_Urban_Continuum_Code` being the second most negative feature shows that there is potentially a pattern with the size and location of counties negatively impacts `Median_Household_Income`. This makes sense because as the code increases, the smaller and more rural the county is. Smaller, rural counties typically have less job opportunities, therefore the household incomes likely are less than larger or more urban counties that have more job opportunities.
    - Interestingly, `Year` still has a relatively strong negative coefficient. This may mean that the baseline median incomes in this dataset trend slightly downward in the later years.
  - The strongest positive impacted features in predicting the `Median_Household_Income` were `Metro` and `Unemployed`
    - `Metro` makes sense because if the county is classified as a metropolitan area, there is likely more job opportunities and higher wages compared to rural or smaller counties. So, if a county is classified as a `Metro`, it will see a positive boost in the prediction formula.
    - `Unemployed`, the total number of people without work, is more interesting that it has a positive coefficient. My theory is that it has proportional to total population. Larger counties with higher income may have more unemployed but that is simply due to the size of the county rather than necessarily a higher `Unemployment_rate`.

Interpretation:

Overall, I do not have a problem with how the LassoCV model got rid of `Civilian_labor_force` and `Employed` in order to predict `Median_Household_Income`. This is because the features are redundant when we have the numbers for people who are `Unemployed` and the `Unemployment_rate`, which captures enough information about the economy to make predictions.

Comparison between the models:
  - Original model with potential data leakage had a lower MAE ($6,432.32 vs. $10,342.01)
  - Original model with potential data leakage had a lower RMSE ($8,849.34 vs. $13,868.24)
  - Original model with potential data leakage had a significantly higher R^2 Score (0.7117 vs. 0.2919)

This shows that while using `Med_HH_Income_Percent_of_State_Total` in the orginal LassoCV model resulted in more accurate predictions and explained a higher proportion of the variance in `Median_Household_Income`, the model essentially was cheating (leaking data) to more accurately predict the target. While R^2 of 0.2919 for the non-data-leaking model seems very low, for predicting household income the errors don't seem significantly higher compared to the model with the data leakage. According to [Duke University](https://people.duke.edu/~rnau/rsquared.htm),"if R-squared is very close to 1, and the data consists of time series, this is usually a bad sign rather than a good one." However when it comes to social science, which socioeconomics can be categorized, a low R-squared is not necessarily bad. A 25% R-squared may actually be considered quite meaningful. In order to further analyze the models, we would need to check for the statistical significance of each feature and consider whether the R-squared of 0.7117 in the original model is outside of the acceptable range for social science machine learning models.

#### Ridge

In [41]:
census = pd.read_csv('..\Processed Data\Census Demographics.csv')
census.head()

,GISJOIN,STATE,STATEFP,STATENH,COUNTY,COUNTYFP,COUNTYNH,total_pop_1970,total_pop_1980,total_pop_1990,total_pop_2000,total_pop_2010,total_pop_2020,male_pop_1970,male_pop_1980,male_pop_1990,male_pop_2000,male_pop_2010,male_pop_2020,female_pop_1970,female_pop_1980,female_pop_1990,female_pop_2000,female_pop_2010,female_pop_2020,pop_under_5_years_1970,pop_under_5_years_1980,pop_under_5_years_1990,pop_under_5_years_2000,pop_under_5_years_2010,pop_under_5_years_2020,pop_5_9_years_1970,pop_5_9_years_1980,pop_5_9_years_1990,pop_5_9_years_2000,pop_5_9_years_2010,pop_5_9_years_2020,pop_10_14_years_1970,pop_10_14_years_1980,pop_10_14_years_1990,...,pop_45_54_years_1990,pop_45_54_years_2000,pop_45_54_years_2010,pop_45_54_years_2020,pop_55_59_years_1970,pop_55_59_years_1980,pop_55_59_years_1990,pop_55_59_years_2000,pop_55_59_years_2010,pop_55_59_years_2020,pop_60_61_years_1970,pop_60_61_years_1980,pop_60_61_years_1990,pop_60_61_years_2000,pop_60_61_years_2010,pop_60_61_years_2020,pop_62_64_years_1970,pop_62_64_years_1980,pop_62_64_years_1990,pop_62_64_years_2000,pop_62_64_years_2010,pop_62_64_years_2020,pop_65_74_years_1970,pop_65_74_years_1980,pop_65_74_years_1990,pop_65_74_years_2000,pop_65_74_years_2010,pop_65_74_years_2020,pop_75_84_years_1970,pop_75_84_years_1980,pop_75_84_years_1990,pop_75_84_years_2000,pop_75_84_years_2010,pop_75_84_years_2020,pop_85_years_and_older_1970,pop_85_years_and_older_1980,pop_85_years_and_older_1990,pop_85_years_and_older_2000,pop_85_years_and_older_2010,pop_85_years_and_older_2020
0,G0100010,Alabama,1,10,Autauga County,1,10,24460.0,32259.0,34222.0,43671.0,54571.0,58805.0,11947.0,15848.0,16653.0,21221.0,26569.0,28390.0,12513.0,16411.0,17569.0,22450.0,28002.0,30415.0,2385.0,2525.0,2679.0,3023.0,3579.0,3513.0,3077.0,2952.0,2752.0,3618.0,3991.0,3796.0,3093.0,3184.0,2887.0,...,3887.0,5635.0,8205.0,7740.0,1029.0,1316.0,1608.0,2291.0,3083.0,4147.0,410.0,511.0,530.0,814.0,1101.0,1545.0,475.0,660.0,775.0,1086.0,1676.0,2079.0,1231.0,1848.0,1966.0,2681.0,4013.0,5420.0,540.0,795.0,1107.0,1342.0,1982.0,2896.0,136.0,181.0,299.0,428.0,551.0,927.0
1,G0100030,Alabama,1,10,Baldwin County,3,30,59382.0,78556.0,98280.0,140415.0,182265.0,231767.0,29032.0,38359.0,47741.0,68848.0,89196.0,112627.0,30350.0,40197.0,50539.0,71567.0,93069.0,119140.0,5081.0,6114.0,6753.0,8621.0,11158.0,11690.0,6302.0,6644.0,7091.0,9486.0,11599.0,13555.0,6769.0,6753.0,7264.0,...,10814.0,19609.0,26921.0,29062.0,3076.0,3979.0,4912.0,8276.0,12523.0,16567.0,1174.0,1604.0,1906.0,2909.0,4800.0,6832.0,1579.0,2198.0,3294.0,4184.0,7212.0,10072.0,4031.0,6447.0,9021.0,12355.0,17803.0,30617.0,1833.0,2726.0,4698.0,7184.0,9532.0,15467.0,469.0,676.0,1160.0,2164.0,3233.0,4606.0
2,G0100050,Alabama,1,10,Barbour County,5,50,22543.0,24756.0,25417.0,29038.0,27457.0,25223.0,10427.0,11668.0,12085.0,14970.0,14576.0,13167.0,12116.0,13088.0,13332.0,14068.0,12881.0,12056.0,1986.0,1958.0,1885.0,1788.0,1702.0,1322.0,2390.0,2181.0,2141.0,2053.0,1642.0,1324.0,2536.0,2305.0,2129.0,...,2334.0,3928.0,4018.0,3191.0,1231.0,1290.0,1114.0,1424.0,1817.0,1834.0,452.0,547.0,441.0,473.0,750.0,721.0,663.0,742.0,685.0,667.0,1049.0,1055.0,1609.0,2110.0,2089.0,2038.0,2238.0,3139.0,673.0,970.0,1303.0,1323.0,1228.0,1472.0,213.0,252.0,334.0,512.0,443.0,492.0
3,G0100070,Alabama,1,10,Bibb County,7,70,13812.0,15723.0,16576.0,20826.0,22915.0,22293.0,6754.0,7628.0,8107.0,10745.0,12301.0,11798.0,7058.0,8095.0,8469.0,10081.0,10614.0,10495.0,1275.0,1244.0,1171.0,1449.0,1378.0,1270.0,1446.0,1395.0,1307.0,1530.0,1405.0,1284.0,1532.0,1485.0,1400.0,...,1766.0,2725.0,3386.0,3107.0,726.0,759.0,709.0,1093.0,1401.0,1618.0,298.0,272.0,288.0,397.0,550.0,606.0,426.0,358.0,404.0,511.0,784.0,820.0,1030.0,1282.0,1108.0,1324.0,1723.0,2152.0,441.0,632.0,773.0,755.0,904.0,1111.0,122.0,149.0,220.0,334.0,279.0,353.0
4,G0100090,Alabama,1,10,Blount County,9,90,26853.0,36459.0,39248.0,51024.0,57322.0,59134.0,13242.0,18043.0,19159.0,25476.0,28362.0,29197.0,13611.0,18416.0,20089.0,25548.0,28960.0,29937.0,2228.0,2554.0,2648.0,3528.0,

With Ridge Regression, it ensures that all coefficients have some weight unlike Lasso Regression. This is important when there may be collinearity but I don't want to choose between the two collinear features (either arbitrarily from the automatic decisions of the model's algorithm or by manually correcting for collinearity). 

Above is a good example with my data to show because a lot of the columns are population totals for certain age groups, sex, and year. Of course when adding these up you get the total population given that year so there will be high collinearity with multiple features in this dataset. Let's run Ridge Regression to apply some weight to all features rather than cancelling some out like what happened with Lasso Regression.

In [44]:
# For these models, let's take in all the historical data for each county in the U.S. to predict the total population of the most recent decade: 2020

X = census.select_dtypes(include="number").drop(columns=['total_pop_2020'])
y = census['total_pop_2020']

ridge_feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

# Testing different alpha scales
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1_000, 10_000]

# Using KFold to have reproducibility with CV since RidgeCV has no random_state
cv_splitter = KFold(n_splits=5, shuffle=True, random_state=seed)

# Ridge is also sensitive to scale, so I need to standardize features first
ridge_cv = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=cv_splitter, scoring='neg_mean_squared_error'))
ridge_cv.fit(X_train, y_train)

# Ridge model from pipeline
ridge_model = ridge_cv.named_steps['ridgecv']

best_ridge_alpha = ridge_model.alpha_
print(f"Optimal Alpha: {best_ridge_alpha}\n")

print(f"Ridge coefficients with lambda = {best_ridge_alpha}:")
for name, coef in zip(ridge_feature_names, ridge_model.coef_):
    if abs(coef) > 1e-8:
        print(f"*** {name}: {coef: .6f}")
    else:
        print(f"    {name}: {coef: .6f} <-- effectively zero")

print("\nNumber of nonzero coefficients:", np.sum(np.abs(ridge_model.coef_) > 1e-8))


# Model evaluation
y_pred_ridge = ridge_cv.predict(X_test)
MAE_ridge = mean_absolute_error(y_test, y_pred_ridge)
RMSE_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
R2_ridge = r2_score(y_test, y_pred_ridge)


print("\nModel Evaluation")
print(f"*** Mean Absolute Error: {MAE_ridge:,.0f} people")
print(f"*** Root Mean Squared Error: {RMSE_ridge:,.0f} people")
print(f"*** R-squared Score: {R2_ridge:.4f}")

Optimal Alpha: 0.01

Ridge coefficients with lambda = 0.01:
*** STATEFP: -10.414082
*** STATENH: -10.414082
*** COUNTYFP:  16.328878
*** COUNTYNH:  16.328878
*** total_pop_1970:  28558.340947
*** total_pop_1980: -3990.558170
*** total_pop_1990:  2291.045832
*** total_pop_2000:  59253.743220
*** total_pop_2010:  204675.307444
*** male_pop_1970: -11768.362492
*** male_pop_1980:  10764.319338
*** male_pop_1990: -15835.358013
*** male_pop_2000:  29249.833084
*** male_pop_2010: -84070.574367
*** male_pop_2020:  124494.759786
*** female_pop_1970: -26222.950255
*** female_pop_1980:  31280.067808
*** female_pop_1990: -1896.034233
*** female_pop_2000:  44267.962466
*** female_pop_2010: -39869.581190
*** female_pop_2020:  45825.644966
*** pop_under_5_years_1970: -617.914155
*** pop_under_5_years_1980:  7321.296297
*** pop_under_5_years_1990: -2762.429311
*** pop_under_5_years_2000:  728.927897
*** pop_under_5_years_2010: -23518.427144
*** pop_under_5_years_2020:  16450.157825
*** pop_5_9_years_1

Results:
  - Of course this model is extremely overfitting the data as it used every feature and has essentially 1 as an R-squared.

As an experiment, let's run the same model again but this time dropping all of the other _2020 features to still try to predict the total populations in 2020 for each county. This would use historical data in order to make predictions for the latest decade. 

In [45]:
columns_to_drop = [col for col in census.columns if '2020' in col]

X = census.select_dtypes(include="number").drop(columns=columns_to_drop)
y = census['total_pop_2020']

ridge_feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

# Testing different alpha scales
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1_000, 10_000]

# Using KFold to have reproducibility with CV since RidgeCV has no random_state
cv_splitter = KFold(n_splits=5, shuffle=True, random_state=seed)

# Ridge is also sensitive to scale, so I need to standardize features first
ridge_cv = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=cv_splitter, scoring='neg_mean_squared_error'))
ridge_cv.fit(X_train, y_train)

# Ridge model from pipeline
ridge_model = ridge_cv.named_steps['ridgecv']

best_ridge_alpha = ridge_model.alpha_
print(f"Optimal Alpha: {best_ridge_alpha}\n")

print(f"Ridge coefficients with lambda = {best_ridge_alpha}:")
for name, coef in zip(ridge_feature_names, ridge_model.coef_):
    if abs(coef) > 1e-8:
        print(f"*** {name}: {coef: .6f}")
    else:
        print(f"    {name}: {coef: .6f} <-- effectively zero")

print("\nNumber of nonzero coefficients:", np.sum(np.abs(ridge_model.coef_) > 1e-8))


# Model evaluation
y_pred_ridge = ridge_cv.predict(X_test)
MAE_ridge = mean_absolute_error(y_test, y_pred_ridge)
RMSE_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
R2_ridge = r2_score(y_test, y_pred_ridge)


print("\nModel Evaluation")
print(f"*** Mean Absolute Error: {MAE_ridge:,.0f} people")
print(f"*** Root Mean Squared Error: {RMSE_ridge:,.0f} people")
print(f"*** R-squared Score: {R2_ridge:.4f}")

Optimal Alpha: 0.1

Ridge coefficients with lambda = 0.1:
*** STATEFP:  236.951726
*** STATENH:  236.951726
*** COUNTYFP:  46.186122
*** COUNTYNH:  46.186122
*** total_pop_1970:  7514.684954
*** total_pop_1980: -19981.791740
*** total_pop_1990:  44521.590246
*** total_pop_2000:  145511.044088
*** total_pop_2010:  76547.648461
*** male_pop_1970: -68460.550866
*** male_pop_1980:  48390.291936
*** male_pop_1990: -11016.070074
*** male_pop_2000: -36837.914244
*** male_pop_2010:  39200.499085
*** female_pop_1970:  4136.738602
*** female_pop_1980:  76674.181003
*** female_pop_1990: -19231.622829
*** female_pop_2000:  34932.700651
*** female_pop_2010:  66245.701231
*** pop_under_5_years_1970:  40840.905159
*** pop_under_5_years_1980:  23315.977906
*** pop_under_5_years_1990: -23845.302528
*** pop_under_5_years_2000: -96459.317194
*** pop_under_5_years_2010:  59793.322054
*** pop_5_9_years_1970: -70448.005593
*** pop_5_9_years_1980: -34944.301505
*** pop_5_9_years_1990: -29713.397830
*** pop_5

Results and Interpretation:

While the errors significantly increased, we see that the R-squared is still essentially 1. By dropping the _2020 features, we eliminated the massive data leak in predicting the `total_pop_2020`. Our errors are large so this shows that it can get relatively close for medium to largely populated counties which shows this model is definitely better in that regard instead of simply memorizing the data and having very low errors. However because R-squared wasn't reduced by much, I can conclude that it is able to explain the scale in counties but not the actual changes. The main issue with these models is that there is severe multicollinearity because the populations are captured multiple times. This results in an overly complex model with very small and large coefficients in order to predict `total_pop_2020`. If we were to use this model on new data, it would likely fail to generalize. 

In order to address the multicollinearity and overfitting, it might be better to create new features for the target. For example, we could engineer the historical growth rates into percentages and force the model to learn more of the patterns that drive growth rather than simply memorizing past population data. Another option is that we could convert the raw data itself into proportions of the total population for that decade while keeping the raw total population numbers. If feature engineering is not the best way, we could reign in the model itself. Given that the alpha 0.1 gave the best predictive score, we could force the alpha to be larger. The tradeoff is that the accuracy will be lower, but the coefficients could result in a more stable, generalized context to allow for new data later on. Finally, the last option is to use Elastic Net Regression which takes both Lasso and Ridge pentalties (L1 and L2 respectively). By finding a balance in the l1_ratio (proportion of Lasso to Ridge), Lasso will zero out features that offer less predictive value while Ridge stabilizes and groups correlated features together.


#### Elastic Net

In [ ]:
# Continuing with same dataset and target to show how Elastic Net could help with this situation

columns_to_drop = [col for col in census.columns if '2020' in col]

X = census.select_dtypes(include="number").drop(columns=columns_to_drop)
y = census['total_pop_2020']

EN_feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

# Adjusted the alphas to be larger in order to avoid overfitting like what we found previously
alphas = [10, 100, 1_000, 10_000, 100_000]
l1_ratios = [0.1, 0.2, 0.3, 0.5]

# Using KFold to have reproducibility with CV since RidgeCV has no random_state
cv_splitter = KFold(n_splits=5, shuffle=True, random_state=seed)

# Ridge and Lasso are both sensitive to scale, so I need to standardize features first
EN_cv = make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=l1_ratios, alphas=alphas, cv=cv_splitter, max_iter=100_000, n_jobs=-1))
EN_cv.fit(X_train, y_train)

# Elastic Net model from pipeline
EN_model = EN_cv.named_steps['elasticnetcv']

best_EN_alpha = EN_model.alpha_
best_EN_l1 = EN_model.l1_ratio_
print(f"Optimal Alpha: {best_EN_alpha}\n")
print(f"Optimal L1 Ratio: {best_EN_l1}\n")

print(f"Ridge coefficients with lambda = {best_EN_alpha}:")
for name, coef in zip(EN_feature_names, EN_model.coef_):
    if abs(coef) > 1e-8:
        print(f"*** {name}: {coef: .6f}")
    else:
        print(f"    {name}: {coef: .6f} <-- zeroed out")

print("\nNumber of nonzero coefficients:", np.sum(np.abs(EN_model.coef_) > 1e-8))


# Model evaluation
y_pred_EN = EN_cv.predict(X_test)
MAE_EN = mean_absolute_error(y_test, y_pred_EN)
RMSE_EN = np.sqrt(mean_squared_error(y_test, y_pred_EN))
R2_EN = r2_score(y_test, y_pred_EN)


print("\nModel Evaluation")
print(f"*** Mean Absolute Error: {MAE_EN:,.0f} people")
print(f"*** Root Mean Squared Error: {RMSE_EN:,.0f} people")
print(f"*** R-squared Score: {R2_EN:.4f}")

Optimal Alpha: 10

Optimal L1 Ratio: 0.7

Ridge coefficients with lambda = 10:
*** STATEFP: -347.329765
*** STATENH: -347.330679
*** COUNTYFP: -89.868308
*** COUNTYNH: -89.868026
*** total_pop_1970:  313.960701
*** total_pop_1980:  1549.889271
*** total_pop_1990:  2921.891023
*** total_pop_2000:  5280.412891
*** total_pop_2010:  6506.422402
*** male_pop_1970:  425.383925
*** male_pop_1980:  1660.230587
*** male_pop_1990:  3011.454054
*** male_pop_2000:  5276.686244
*** male_pop_2010:  6324.167145
*** female_pop_1970:  270.223366
*** female_pop_1980:  1395.856747
*** female_pop_1990:  2795.924950
*** female_pop_2000:  4942.362111
*** female_pop_2010:  6152.869165
*** pop_under_5_years_1970:  603.067852
*** pop_under_5_years_1980:  1854.288452
*** pop_under_5_years_1990:  3117.914031
*** pop_under_5_years_2000:  5460.530330
*** pop_under_5_years_2010:  7188.733200
*** pop_5_9_years_1970:  686.961708
*** pop_5_9_years_1980:  1767.845870
*** pop_5_9_years_1990:  3266.146295
*** pop_5_9_yea

Results and Interpretations:
  - Even after playing around with the ranges for alpha and l1_ratio, we are still capturing almost all of the variance. The tradeoff I found when messing with the hyperparameter ranges is that if you increase alpha and decrease l1_ratio they both lead to a lower R-squared but significantly higher errors. This also shows that there needs to be either more precise feature selection first before building the final model or to do feature engineering in order to have more precise predictions while generalizing. To show this, I will do some feature engineering in attempt to create a more generalized model while having more acceptable errors (finding the balance in errors for small, medium, and large populations).

In [52]:
census_engineered = census.copy()

# New target: growth rate from 2010 to 2020
census_engineered['recent_growth_rate'] = (census_engineered['total_pop_2020'] - census_engineered['total_pop_2010']) / census_engineered['total_pop_2010']

# converting 2010 features into proportions
cols_2010 = [col for col in census_engineered.columns if '2010' in col and col != 'total_pop_2010']

for col in cols_2010:
    new_col_name = col.replace('pop_', 'pct_')
    census_engineered[new_col_name] = census_engineered[col] / census_engineered['total_pop_2010']

# dropping features that would leak information
cols_to_drop = [col for col in census_engineered.columns if '2020' in col or '19' in col or '2000' in col or col == 'total_pop_2010']
cols_to_drop.extend(cols_2010)

census_engineered = census_engineered.drop(columns=cols_to_drop)
census_engineered.head()

,GISJOIN,STATE,STATEFP,STATENH,COUNTY,COUNTYFP,COUNTYNH,recent_growth_rate,male_pct_2010,female_pct_2010,pct_under_5_years_2010,pct_5_9_years_2010,pct_10_14_years_2010,pct_15_17_years_2010,pct_20_years_2010,pct_21_years_2010,pct_22_24_years_2010,pct_25_29_years_2010,pct_30_34_years_2010,pct_35_44_years_2010,pct_45_54_years_2010,pct_55_59_years_2010,pct_60_61_years_2010,pct_62_64_years_2010,pct_65_74_years_2010,pct_75_84_years_2010,pct_85_years_and_older_2010
0,G0100010,Alabama,1,10,Autauga County,1,10,0.077587,0.486870,0.513130,0.065584,0.073134,0.078613,0.050448,0.012186,0.010738,0.033516,0.057851,0.061021,0.151051,0.150355,0.056495,0.020176,0.030712,0.073537,0.036320,0.010097
1,G0100030,Alabama,1,10,Baldwin County,3,30,0.271594,0.489375,0.510625,0.061219,0.063638,0.065432,0.039585,0.010743,0.009579,0.031520,0.056220,0.058755,0.129224,0.147703,0.068708,0.026335,0.039569,0.097676,0.052297,0.017738
2,G0100050,Alabama,1,10,Barbour County,5,50,-0.081364,0.530866,0.469134,0.061988,0.059803,0.058237,0.039043,0.012674,0.012383,0.040281,0.073205,0.065848,0.132134,0.146338,0.066176,0.027315,0.038205,0.081509,0.044724,0.016134
3,G0100070,Alabama,1,10,Bibb County,7,70,-0.027144,0.536810,0.463190,0.060135,0.061314,0.062841,0.042679,0.011346,0.012874,0.040847,0.069954,0.071831,0.147589,0.147763,0.061139,0.024002,0.034213,0.075191,0.039450,0.012175
4,G0100090,Alabama,1,10,Blount County,9,90,0.031611,0.494784,0.505216,0.063082,0.067688,0.071247,0.044067,0.011427,0.010048,0.032727,0.059942,0.060029,0.138254,0.141045,0.064443,0.025679,0.036810,0.088605,0.044520,0.014096


In [59]:
X_engineered = census_engineered.select_dtypes(include="number").drop(columns=['recent_growth_rate'])
y_engineered = census_engineered['recent_growth_rate']

EN_feature_names = X_engineered.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X_engineered, y_engineered, test_size=0.2, random_state=seed)

alphas = [0.00001, 0.0001, 0.001, 0.01, 1]
# l1_ratios = [0.01, 0.05, 0.1, 0.25, 0.4]
l1_ratios = [0.01, 0.05, 0.1, 0.25, 0.4, 0.5, 0.75, 0.9]


# Using KFold to have reproducibility with CV since RidgeCV has no random_state
cv_splitter = KFold(n_splits=5, shuffle=True, random_state=seed)

# Ridge and Lasso are both sensitive to scale, so I need to standardize features first
EN_cv = make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=l1_ratios, alphas=alphas, cv=cv_splitter, max_iter=100_000, n_jobs=-1))
EN_cv.fit(X_train, y_train)

# Elastic Net model from pipeline
EN_model = EN_cv.named_steps['elasticnetcv']

best_EN_alpha = EN_model.alpha_
best_EN_l1 = EN_model.l1_ratio_
print(f"Optimal Alpha: {best_EN_alpha}\n")
print(f"Optimal L1 Ratio: {best_EN_l1}\n")

print(f"Ridge coefficients with lambda = {best_EN_alpha}:")
for name, coef in zip(EN_feature_names, EN_model.coef_):
    if abs(coef) > 1e-8:
        print(f"*** {name}: {coef: .6f} <-- nonzero")
    else:
        print(f"    {name}: {coef: .6f}")

print("\nNumber of nonzero coefficients:", np.sum(np.abs(EN_model.coef_) > 1e-8))


# Model evaluation
y_pred_EN = EN_cv.predict(X_test)
MAE_EN = mean_absolute_error(y_test, y_pred_EN)
RMSE_EN = np.sqrt(mean_squared_error(y_test, y_pred_EN))
R2_EN = r2_score(y_test, y_pred_EN)


print("\nModel Evaluation")
print(f"*** Mean Absolute Error: {MAE_EN * 100:.2f}%")
print(f"*** Root Mean Squared Error: {RMSE_EN * 100:.2f}%")
print(f"*** R-squared Score: {R2_EN:.4f}")

Optimal Alpha: 0.01

Optimal L1 Ratio: 0.5

Ridge coefficients with lambda = 0.01:
    STATEFP: -0.000000
    STATENH: -0.000000
*** COUNTYFP: -0.001586 <-- nonzero
*** COUNTYNH: -0.000721 <-- nonzero
*** male_pct_2010: -0.001558 <-- nonzero
*** female_pct_2010: -0.029236 <-- nonzero
*** pct_under_5_years_2010:  0.129714 <-- nonzero
    pct_5_9_years_2010: -0.000000
    pct_10_14_years_2010:  0.000000
    pct_15_17_years_2010: -0.000000
*** pct_20_years_2010:  0.002834 <-- nonzero
*** pct_21_years_2010:  0.048720 <-- nonzero
    pct_22_24_years_2010: -0.000000
*** pct_25_29_years_2010: -0.062366 <-- nonzero
*** pct_30_34_years_2010: -0.004241 <-- nonzero
*** pct_35_44_years_2010:  0.036699 <-- nonzero
*** pct_45_54_years_2010:  0.007256 <-- nonzero
    pct_55_59_years_2010: -0.000000
    pct_60_61_years_2010: -0.000000
*** pct_62_64_years_2010:  0.031681 <-- nonzero
    pct_65_74_years_2010:  0.000000
    pct_75_84_years_2010:  0.000000
    pct_85_years_and_older_2010: -0.000000

Numbe

Results:
  - After reducing the alphas and l1_ratios I was able to create a more generalized model in predicting the growth rates of the total population from 2010 to 2020. We have a lower R-squared of 0.5328, which is an improvement from the near perfect R-squared scores from the previous models.
  - `pct_under_5_years_2010`, `pct_21_years_2010`, and `pct_35_44_years_2010` had the strongest positive coefficients.
    - This could mean that the proportion of young children, young adults, and people in their 30s-40s represent strong predictors because of they could be core age groups for county growth in the U.S. This could potentially represent parents and young children in growing populations as well as university and entry-level working-age people going to counties with opportunities. `pct_62_64_years_2010` is also another strong predictor as people nearing retirement age may move around to certain areas to retire. Besides thinking about people moving around, it makes sense that young families make up the strongest positive coefficients in predicting population growth as they are the age group having children which directly affects the population growth rate. The ages in between 22 and 34 is interesting as they are somewhat strong negative coefficients. This could be because this age group moves around more so it affects the county populations in the negative direction. 

Interpretation:

Overall, this model definitely generalizes better rather than overfitting because of the lower R-squared, elimination of some features, and the low but significant error rates. The balancing act was finding the right alpha and l1_ratio ranges. When alpha is low, it means the model is less restricted from penalization so it memorizes more of the training data. In contrast, when it is high there is more penalty for using features which results in underfitting. Now, the model has a more realistic R-squared score. In the previous models with near-perfect R-squared scores, they were artificially inflated by the raw population counts which resulted in capturing the variance in scale across U.S. counties (small counties remain small and large counties remain large). This combined with my feature engineering resulted in a more generalized model that provides valuable insights without simply memorizing the data with overly complicated coefficients.